# Arabic Costs Double on AI — the receipt

> **The same sentence costs 2× more tokens in Arabic than in English on AI.**
> Not an opinion. The mechanical result of two design choices made in 1992 and 2019.
> Affects every Arabic-script language — Darija, Algerian, Egyptian, MSA, Persian, Urdu —
> across **ChatGPT, Claude, Gemini, and every major LLM**.

**Author:** [Miloud Belarebia](https://github.com/miloudbelarebia) — [databelarebia.com](https://databelarebia.com)
**License:** MIT
**Source data:** [`github.com/openai/tiktoken`](https://github.com/openai/tiktoken) — vocab `o200k_base`

---

## What this notebook does

1. Downloads the **real** vocabulary file shipped by OpenAI (used by GPT-4o, GPT-4.1, GPT-5, o1, o3).
2. Classifies all **200,019 tokens** by Unicode script (Arabic / Latin / CJK / digit / other).
3. Counts how many tokens are dedicated to each writing system.
4. Measures the actual cost of writing the same sentence in English vs. Arabic.

Everything is reproducible. Re-run the notebook, you get the same numbers.

> **Why OpenAI as the example?** Because OpenAI's tokenizer is the only one of the big three
> (ChatGPT / Claude / Gemini) that is **publicly downloadable and byte-for-byte verifiable**.
> Claude and Gemini use proprietary tokenizers, but built on the same UTF-8 foundation and
> trained on similarly English-dominant corpora — the **mechanism is identical**, only the
> exact numbers vary.

## 0. Why does this happen? A 32-year-old design story

This isn't a bug or malicious choice. It's the **stacked consequence of two design choices** —
one from 1992, one from 2019. Both were defensible at the time. Neither was malicious. But
together, they produce today's tax on Arabic-script users.

### 🧱 Layer 1 — UTF-8 (1992): the foundation

UTF-8 is the standard that decides how text is stored as bytes on disk. Every text file,
every webpage, every API call uses it. It was designed by **Ken Thompson and Rob Pike in
September 1992**, on a placemat in a New Jersey diner.

They had to stay 100% backward-compatible with ASCII (the old 128-character English-only
standard). So the design was:

| Unicode range | Byte cost | Examples |
|---|---:|---|
| U+0000 – U+007F | **1 byte** | `A`, `z`, digits, basic Latin |
| U+0080 – U+07FF | **2 bytes** | `é`, Greek, Cyrillic, Hebrew, **Arabic (U+0600–U+06FF)** |
| U+0800 – U+FFFF | **3 bytes** | Chinese 中, Japanese, Korean |
| U+10000+ | **4 bytes** | Emojis 😀, rare scripts |

→ **An Arabic letter takes 2 bytes. A Latin letter takes 1 byte.** This is the
[RFC 3629 specification](https://www.rfc-editor.org/rfc/rfc3629), not anyone's opinion.
Before any AI exists. Before any tokenizer is trained.

### 🤖 Layer 2 — The tokenizer (2019–2024): the amplifier

In 2019, OpenAI's GPT-2 introduced **byte-level BPE**: the tokenizer starts from raw
UTF-8 bytes (256 of them) and learns which sequences appear often in training data,
promoting them to single tokens. On paper, this is pro-multilingual — every character on
Earth is encodable.

But "universal coverage" ≠ "equal cost". BPE only learns good groupings for what it has
seen a lot. OpenAI's training corpus is overwhelmingly English and code. So English words
like `function`, `the`, `intelligence` become single tokens; Arabic words like
**الذكاء** (intelligence) stay fragmented into 3–5 sub-tokens each.

**The UTF-8 handicap (×2 bytes) isn't corrected by the BPE — it's inherited and then
amplified by under-representation in the training data.**

Evidence: OpenAI keeps **doubling the vocabulary** to soften the symptom.

| Year | Model | Tokenizer | Vocab size |
|---|---|---|---:|
| 2019 | GPT-2 | r50k_base | ~50,257 |
| 2023 | GPT-3.5 / GPT-4 | `cl100k_base` | 100,256 |
| 2024+ | GPT-4o, o1, o3 | **`o200k_base`** | **200,019** |

The same Chinese text took 12 tokens in GPT-4 and only 2 tokens in GPT-4o — purely thanks
to the larger vocab. They are buying space to atténuate non-English fragmentation, never
fixing the root cause. **Even `o200k_base` — the biggest yet — still gives Arabic only 4.0%
of its dictionary.** This notebook proves it.

> **Methodological note.** The facts above (UTF-8 byte sizes, BPE mechanism, vocabulary
> sizes) are documented in public sources. The framing — calling this a "tax" or "structural
> injustice", and reading the vocab expansions as an implicit admission of bias — is *my
> interpretation*, supported by the measurements you're about to see, not a statement
> from OpenAI.

## 1. Install dependencies

Only one library is needed: `tiktoken`, OpenAI's official tokenizer.
For the visualization at the end, we also use `matplotlib`, `arabic-reshaper`
and `python-bidi` to render Arabic script correctly.

In [ ]:
# Run this once. After that, the kernel has everything it needs.
%pip install -q tiktoken matplotlib arabic-reshaper python-bidi

## 2. Load OpenAI's real tokenizer

`o200k_base` is the byte-pair-encoding (BPE) vocabulary used by every recent
OpenAI model: GPT-4o, GPT-4o mini, GPT-4.1, GPT-5, and the o1 / o3 / o4 reasoning
family. It is a list of 200,019 *tokens* (sub-word pieces). Each piece of text
the model reads or writes is decomposed into these tokens.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
N = enc.n_vocab
print(f"Loaded o200k_base — {N:,} tokens in vocabulary")

## 3. Classify each token by Unicode script

We look at every token, decode its bytes to a string, and detect whether
its letters are mostly **Arabic**, **Latin**, **CJK** (Chinese/Japanese/Korean),
or **digits / other**.

Before running it on the real vocabulary, the function is tested against
14 hand-picked cases to make sure the classifier is sound.

In [ ]:
import unicodedata

def classify_token(s: str) -> str:
    """Classify a decoded token by the dominant Unicode script of its letters."""
    counts = {"arabic": 0, "latin": 0, "cjk": 0}
    has_letter = False
    for ch in s:
        if not ch.isalpha():
            continue
        has_letter = True
        try:
            name = unicodedata.name(ch)
        except ValueError:
            continue
        if "ARABIC" in name:
            counts["arabic"] += 1
        elif "LATIN" in name:
            counts["latin"] += 1
        elif ("CJK" in name or "HIRAGANA" in name or "KATAKANA" in name
              or "HANGUL" in name):
            counts["cjk"] += 1
    if not has_letter:
        if any(c.isdigit() for c in s):
            return "digit"
        return "other"
    dom = max(counts, key=counts.get)
    return dom if counts[dom] > 0 else "other"


# --- sanity check on 14 known cases ---
cases = [
    ("ال", "arabic"), ("كيتعلم", "arabic"), (" السلام", "arabic"),
    ("hello", "latin"), (" the", "latin"), ("ization", "latin"),
    ("中文", "cjk"), ("こんにちは", "cjk"), ("한국어", "cjk"),
    ("123", "digit"), ("   ", "other"), ("===", "other"),
    ("def", "latin"), ("café", "latin"),
]
ok = sum(classify_token(t) == e for t, e in cases)
print(f"Sanity check: {ok}/{len(cases)} cases pass")
assert ok == len(cases), "Classifier failed sanity check"


## 4. Walk the full vocabulary

We iterate over all 200,019 token ids, decode each one, run the classifier,
and tally the result. Takes about a second.

In [ ]:
from collections import Counter

buckets = Counter()
arabic_examples = []

for tok_id in range(N):
    try:
        text = enc.decode_single_token_bytes(tok_id).decode("utf-8")
    except Exception:
        buckets["undecodable"] += 1
        continue
    cat = classify_token(text)
    buckets[cat] += 1
    if cat == "arabic" and len(arabic_examples) < 20:
        arabic_examples.append(text)

print(f"Done — counted {sum(buckets.values()):,} tokens")


In [ ]:
# --- pretty breakdown ---
order = ["latin", "cjk", "arabic", "digit", "other", "undecodable"]
labels = {
    "latin":       "Latin (English, code, European languages)",
    "cjk":         "CJK (Chinese / Japanese / Korean)",
    "arabic":      "Arabic (all variants: MSA, Darija, Persian, Urdu)",
    "digit":       "Digits",
    "other":       "Other (punctuation, symbols, spaces)",
    "undecodable": "Undecodable (raw byte fragments)",
}

print(f"{'Script':<55} {'Tokens':>10} {'%':>7}")
print("-" * 75)
for k in order:
    n = buckets.get(k, 0)
    pct = 100 * n / N
    print(f"{labels[k]:<55} {n:>10,} {pct:>6.2f}%")

print()
arab = buckets.get("arabic", 0)
latin = buckets.get("latin", 0)
print(f"➜ Latin has {latin/arab:.0f}x more tokens than Arabic.")


In [ ]:
# --- examples of Arabic tokens that ARE in the vocabulary ---
print("First Arabic tokens found in o200k_base:")
print("   ", "  ".join(arabic_examples))


### What we see so far

Out of **200,019 tokens**:

| Script | Tokens | % |
|---|---:|---:|
| **Latin** (English + code + European languages) | ~134,868 | **67.4%** |
| **Arabic** (all variants combined) | ~7,964 | **4.0%** |

The Latin script gets **17x more dedicated tokens** than the Arabic script.

This isn't a bug or a malicious choice — it's the mechanical result of training
the BPE algorithm on a corpus that's overwhelmingly English and code. The
algorithm gives a token to whatever it sees often. It rarely sees Arabic, so
Arabic gets rare tokens.

The **direct consequence**: any sentence in Arabic — and especially in **modern,
technical Arabic** (Darija, Algerian, Egyptian, MSA used for tech writing) —
needs to be broken up into many small sub-tokens, because the full Arabic
words aren't in the vocabulary.

Let's measure exactly how much that costs.

## 5. Measure the Arabic tax on real sentences

We tokenize the same idea in English and in Arabic and compare the token counts.

In [ ]:
pairs = [
    ("Hello, how are you today?",        "السلام عليكم كيف داير اليوم"),
    ("Artificial intelligence is here", "الذكاء الاصطناعي وصل"),
    ("The model learns from data",      "الموديل كيتعلم من الداتا"),
]

print(f"{'EN tok':>7} | {'AR tok':>7} | {'Tax':>6}  | Arabic sentence")
print("-" * 75)
tot_en = tot_ar = 0
for en, ar in pairs:
    ne = len(enc.encode(en))
    na = len(enc.encode(ar))
    tot_en += ne
    tot_ar += na
    print(f"{ne:>7d} | {na:>7d} | {na/ne:5.2f}x | {ar}")

print("-" * 75)
print(f"{tot_en:>7d} | {tot_ar:>7d} | {tot_ar/tot_en:5.2f}x | TOTAL")
print()
print(f"→ Average: Arabic takes {tot_ar/tot_en:.2f}x more tokens than English.")
print(f"  That's roughly +{int(round((tot_ar/tot_en - 1)*100))}% more, "
      f"to express the same idea.")


### An important nuance

Notice that the **first** sentence ("السلام عليكم...") is actually slightly
**cheaper** in Arabic (6 tokens vs 7 in English). That's because it's a
canonical religious greeting — massively present in the training corpus —
so the tokenizer learned it as whole pieces.

The tax explodes the moment we switch to **modern, technical Arabic**:

- "الذكاء الاصطناعي وصل" (*AI is here*) → **+100% (double)**
- "الموديل كيتعلم من الداتا" (*the model learns from data*) → +50%

Words like *data*, *model*, *AI* exist as single tokens in English. They
don't exist at all in Arabic. So every time you write tech content in any
Arabic-script dialect — Moroccan Darija, Algerian, Egyptian, Levantine,
or MSA — you pay double in API cost, context window, and latency.

## 6. Generate the infographic

A single image summarising everything we just measured — square 1080×1080,
ready for social media.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import arabic_reshaper
from bidi.algorithm import get_display

# --- numbers we just measured ---
EN_TXT, AR_TXT = "Artificial intelligence is here", "الذكاء الاصطناعي وصل"
EN_TOK, AR_TOK = 4, 8
TAXE_PCT = int(round((AR_TOK/EN_TOK - 1) * 100))   # 100
LATIN_N, ARAB_N = buckets.get("latin", 0), buckets.get("arabic", 0)
LATIN_PCT = 100 * LATIN_N / N
ARAB_PCT  = 100 * ARAB_N  / N
RATIO     = round(LATIN_N / ARAB_N)

# --- soft, paper-like palette ---
BG, CARD, BORDER = "#FAF6EF", "#FFFFFF", "#E6DFD0"
FG, MUTED, DIM = "#2D3748", "#6B7280", "#9CA3AF"
ACCENT, LATIN_C = "#C5635F", "#5B82C4"
arab_font = "Geeza Pro"     # macOS — change to "Noto Sans Arabic" on Linux

def ar(s):
    return get_display(arabic_reshaper.reshape(s))

def thou(n):
    return f"{n:,}".replace(",", " ")

fig = plt.figure(figsize=(1080/150, 1080/150), dpi=150, facecolor=BG)
ax = fig.add_axes((0, 0, 1, 1)); ax.set_facecolor(BG); ax.axis("off")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
LEFT, RIGHT = 0.06, 0.94
WIDTH = RIGHT - LEFT

# Header — universalised: "Arabic Tax" covers all Arabic-script dialects
ax.text(LEFT, 0.965, "ARABIC COSTS DOUBLE ON AI", fontsize=28, fontweight="bold",
        color=ACCENT, ha="left", va="top")
ax.text(LEFT, 0.890, "on ChatGPT, Claude & Gemini — the same sentence, twice the tokens.",
        fontsize=15.5, color=FG, ha="left", va="top")
ax.text(LEFT, 0.845,
        "Darija · Algerian · Egyptian · MSA … — same problem.",
        fontsize=12.5, color=MUTED, ha="left", va="top", fontstyle="italic")
ax.plot([LEFT, RIGHT], [0.800, 0.800], color=BORDER, linewidth=1)

# Demo (same idea, two languages)
ax.text(LEFT, 0.770, "Same idea, two languages:", fontsize=13, color=MUTED,
        ha="left", va="top", fontweight="600")
BAR_X0, BAR_X1 = LEFT, RIGHT - 0.20
BAR_W = BAR_X1 - BAR_X0
max_tok = max(EN_TOK, AR_TOK)
bar_h = 0.032

ax.text(LEFT, 0.722, EN_TXT, fontsize=19, color=FG, ha="left", va="center")
w_en = BAR_W * (EN_TOK/max_tok)
ax.add_patch(patches.FancyBboxPatch((BAR_X0, 0.677-bar_h/2), w_en, bar_h,
    boxstyle="round,pad=0,rounding_size=0.006", facecolor=LATIN_C, edgecolor="none"))
ax.text(BAR_X0+w_en+0.014, 0.677, f"{EN_TOK} tokens", fontsize=15,
        color=LATIN_C, ha="left", va="center", fontweight="bold")

ax.text(BAR_X1, 0.614, ar(AR_TXT), fontsize=26, color=FG, ha="right", va="center",
        fontname=arab_font)
w_ar = BAR_W * (AR_TOK/max_tok)
ax.add_patch(patches.FancyBboxPatch((BAR_X0, 0.569-bar_h/2), w_ar, bar_h,
    boxstyle="round,pad=0,rounding_size=0.006", facecolor=ACCENT, edgecolor="none"))
ax.text(BAR_X0+w_ar+0.014, 0.569, f"{AR_TOK} tokens", fontsize=15,
        color=ACCENT, ha="left", va="center", fontweight="bold")

# Hero card — uniform 30px gap above (after Bar AR) and below (before WHY?)
ax.add_patch(patches.FancyBboxPatch((LEFT+0.004, 0.356), WIDTH, 0.170,
    boxstyle="round,pad=0,rounding_size=0.025", facecolor="#EFE7D6", edgecolor="none"))
ax.add_patch(patches.FancyBboxPatch((LEFT, 0.362), WIDTH, 0.170,
    boxstyle="round,pad=0,rounding_size=0.025", facecolor=CARD,
    edgecolor=BORDER, linewidth=1.2))
ax.text(0.5, 0.472, f"+{TAXE_PCT}%", fontsize=60, fontweight="bold",
        color=ACCENT, ha="center", va="center")
ax.text(0.5, 0.390, "more tokens to say the exact same thing in Arabic",
        fontsize=13.5, color=FG, ha="center", va="center")

# Why (bars only — context is in the header)
ax.text(LEFT, 0.332, "WHY?", fontsize=13, color=MUTED, ha="left",
        va="top", fontweight="bold")

BAR2_X0 = LEFT + 0.085          # bars start after the label column
BAR2_X1 = RIGHT - 0.18
BAR2_W  = BAR2_X1 - BAR2_X0
SCALE = 70

ax.text(LEFT, 0.272, "Latin", fontsize=14, color=LATIN_C, ha="left",
        va="center", fontweight="bold")
w_lat = BAR2_W * (LATIN_PCT/SCALE)
ax.add_patch(patches.Rectangle((BAR2_X0, 0.272-0.019), w_lat, 0.038,
    facecolor=LATIN_C, edgecolor="none"))
ax.text(BAR2_X0+w_lat+0.012, 0.272, f"{LATIN_PCT:.1f}%", fontsize=14,
        color=LATIN_C, ha="left", va="center", fontweight="bold")

ax.text(LEFT, 0.217, "Arabic", fontsize=14, color=ACCENT, ha="left",
        va="center", fontweight="bold")
w_arb = max(BAR2_W * (ARAB_PCT/SCALE), 0.010)
ax.add_patch(patches.Rectangle((BAR2_X0, 0.217-0.019), w_arb, 0.038,
    facecolor=ACCENT, edgecolor="none"))
ax.text(BAR2_X0+w_arb+0.012, 0.217, f"{ARAB_PCT:.1f}%", fontsize=14,
        color=ACCENT, ha="left", va="center", fontweight="bold")

ax.text(LEFT, 0.162, f"That's {RATIO}× more Latin tokens than Arabic.",
        fontsize=16, color=FG, ha="left", va="top", fontweight="700")
ax.text(LEFT, 0.122, "Higher API cost. Smaller context window. Fragmented text.",
        fontsize=11.5, color=MUTED, ha="left", va="top")

# Footer
ax.plot([LEFT, RIGHT], [0.077, 0.077], color=BORDER, linewidth=1)
ax.text(LEFT, 0.044, "Data source: github.com/openai/tiktoken  ·  vocab o200k_base",
        fontsize=10, color=MUTED, ha="left", va="center")
ax.text(RIGHT, 0.044, "databelarebia.com", fontsize=10.5, color=FG, ha="right",
        va="center", fontweight="bold")
ax.text(LEFT, 0.014, "Reproducible notebook: github.com/miloudbelarebia/arabic-tax",
        fontsize=9.5, color=ACCENT, ha="left", va="center", fontweight="600")
ax.text(RIGHT, 0.014, "Miloud Belarebia", fontsize=10, color=MUTED,
        ha="right", va="center")

fig.savefig("the-arabic-tax.png", facecolor=BG, dpi=150)
plt.show()
print("Saved: the-arabic-tax.png")


## 7. Conclusion

The takeaway in one sentence: **the same idea costs 2× more tokens in
Arabic than in English on ChatGPT, because OpenAI's vocabulary dedicates
17× more tokens to Latin script than to Arabic.**

This isn't ChatGPT-specific. Claude (Anthropic) and Gemini (Google) train
their tokenizers on similar English-dominant corpora — so the tax pattern
holds across all major LLMs, even if the exact numbers vary.

This isn't a Darija problem either. Every dialect written in Arabic letters
— Moroccan Darija, Algerian, Tunisian, Egyptian, Levantine, Gulf, MSA — and
every Arabic-script language (Persian, Urdu, Pashto) pays the same
structural tax. ~500 million people, one shared bill.

### What this means in practice

- **Higher API bills** for any product that serves Arabic-speaking users.
- **Smaller usable context window** in Arabic — a 128k-token window shrinks
  to roughly 64k of useful content.
- **Worse model behavior**, because the model is reading fragmented words
  instead of whole units.
- **A linguistic-sovereignty issue**: the dominant AI infrastructure
  represents some languages as first-class and others as expensive afterthoughts.

### How to use this notebook

- Run it once. Confirm the numbers.
- Fork it. Swap the example sentences for your own (your real Arabic text,
  not toy examples — the tax climbs higher with technical vocabulary).
- Cite it. The data source is OpenAI's official `tiktoken` repository —
  anyone can verify.

---

**Author:** Miloud Belarebia — [@miloudbelarebia](https://github.com/miloudbelarebia)
**Website:** [databelarebia.com](https://databelarebia.com)
**License:** MIT — use, fork, share.